In [ ]:
# Install dependencies
!pip install timm diffusers


In [ ]:
# IMPORTANT: This notebook is designed for YOUR CUSTOM DiT fork with x2 modifications.
# 
# Option 1: If you've pushed your code to GitHub, replace the URL below:
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# %cd YOUR_REPO
#
# Option 2: For now, we clone the base DiT repo and overwrite with your custom files:
!git clone -b vit-side https://github.com/mrdjango/DuoDiT.git
%cd DiT
!git checkout vit-side
!git pull

# The next cells will overwrite models.py (with x2 modifications) and add new scripts


In [ ]:
# Download ImageNet Validation Dataset (ILSVRC2012)
# This will download ~6.3GB and organize it into class folders

import os
import tarfile
from pathlib import Path
import xml.etree.ElementTree as ET

# Download validation set
print('Downloading ImageNet validation set (~6.3GB)...')
!wget -nc https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar

# Download validation ground truth annotations
print('Downloading validation ground truth...')
!wget -nc https://image-net.org/data/ILSVRC/2012/ILSVRC2012_devkit_t12.tar.gz

# Extract validation images
print('Extracting validation images...')
os.makedirs('imagenet_val', exist_ok=True)
!tar -xf ILSVRC2012_img_val.tar -C imagenet_val

# Extract devkit to get ground truth
print('Extracting devkit...')
!tar -xzf ILSVRC2012_devkit_t12.tar.gz

# Parse ground truth from devkit
print('Parsing validation ground truth...')
gt_file = 'ILSVRC2012_devkit_t12/data/ILSVRC2012_validation_ground_truth.txt'
with open(gt_file, 'r') as f:
    # Ground truth file has 1-indexed class IDs, convert to 0-indexed
    labels = [int(line.strip()) - 1 for line in f]

# Organize into class folders
print('Organizing into class folders...')
val_dir = Path('imagenet_val')
organized_dir = Path('imagenet_val_organized')

# Create class directories
for class_id in set(labels):
    (organized_dir / str(class_id)).mkdir(parents=True, exist_ok=True)

# Move images to class folders
val_images = sorted(val_dir.glob('ILSVRC2012_val_*.JPEG'))
for idx, img_path in enumerate(val_images):
    class_id = labels[idx]
    target_path = organized_dir / str(class_id) / img_path.name
    img_path.rename(target_path)

print(f'✓ ImageNet validation set organized into {len(set(labels))} class folders')
print(f'Total images: {len(val_images)}')
print('Dataset ready at: ./imagenet_val_organized')


In [ ]:
# Custom results directory
!torchrun --nnodes=1 --nproc_per_node=1 train_x2_finetune.py \
    --model DiT-XL/2 \
    --data-path ./imagenet_val_organized \
    --results-dir /content/drive/MyDrive/dit_checkpoints \
    --classes 0 1 2 3 4 5 6 7 8 9 \
    --epochs 2 \
    --global-batch-size 50 \
    --log-every 10 \
    --ckpt-every 100
